# Polars

Red and white wine-quality tables need to be combined and compared. Decide which columns identify the wine type, how quality groups differ in size, and when a summary should be computed. Use expressions and query plans to connect the analytical question to the work performed.

Polars is a fast DataFrame library for tabular data cleaning, transformation, and analysis.


## 1. Install and import

If Polars is not installed in the notebook's selected environment, run this once in a code cell:

```python
%pip install polars matplotlib
```

Polars uses expressions such as `pl.col("alcohol")` to describe transformations. Those expressions are evaluated by a DataFrame or LazyFrame.

In [ ]:
import io
from dataset_cache import get_dataset

import matplotlib.pyplot as plt
import polars as pl

print(f"Polars version: {pl.__version__}")

## 2. Check whether the two files can be compared

Inspect schemas and provenance before combining the files. Preserve the wine-type label so that a pooled summary does not conceal group differences. The downloaded observations support description, not a randomized comparison of wine types.

The UCI dataset contains separate semicolon-separated files for red and white vinho verde wines. Each row has physicochemical measurements and a sensory quality score from 0 to 10. We add a `wine_type` column before combining them. The data is real and openly available; it is not synthetic.

Downloads are cached in `data/cache/`. Later runs reuse the local files; see the README for offline setup.


In [ ]:
def load_wine_table(filename, wine_type):
    return (
        pl.read_csv(
            get_dataset(filename),
            separator=";",
            infer_schema_length=10_000,
        )
        .with_columns(pl.lit(wine_type).alias("wine_type"))
    )

red_wine = load_wine_table("winequality-red.csv", "red")
white_wine = load_wine_table("winequality-white.csv", "white")
wine = pl.concat([red_wine, white_wine], how="vertical")

print(f"Rows: {wine.height:,}")
print(f"Columns: {wine.width}")
wine.head()

## 3. Inspect and transform with expressions

Before changing the wine table, notice that Polars does not ask us to write a loop over rows. Instead, we describe a calculation for a column, and Polars applies it to the column's values.

| Expression | Meaning in this example | What it produces |
| --- | --- | --- |
| `pl.col("alcohol")` | Refer to the alcohol column | A Polars expression, not a Python list |
| `pl.col("alcohol") / 100` | Divide each alcohol value by 100 | A derived value for every row |
| `.alias("alcohol_fraction")` | Give the derived column a name | A readable output column |
| `.with_columns(...)` | Add or replace columns | A DataFrame with the original rows retained |

<div style="background-color:#eaf4fb; border-left:5px solid #2878a5; padding:10px 14px; margin:12px 0;">
<strong>Think of an expression as a recipe for a column.</strong><br>
`pl.col("alcohol") / 100` describes what to do to each alcohol value. It does not extract a single value immediately. The DataFrame operation, such as `with_columns`, supplies the table context where that recipe is evaluated.
</div>

`select` chooses or creates columns. `with_columns` adds or replaces columns while keeping the other columns. Expressions can be composed, so the same vocabulary works for simple and complex transformations. `describe` is a quick diagnostic, not a complete analysis of data quality.

In [ ]:
wine = wine.with_columns(
    (pl.col("alcohol") / 100).alias("alcohol_fraction"),
    pl.when(pl.col("quality") >= 7)
    .then(pl.lit("high"))
    .otherwise(pl.lit("standard"))
    .alias("quality_band"),
)

print(wine.select(["wine_type", "quality", "alcohol", "quality_band"]).head())
wine.select(["quality", "alcohol", "pH", "volatile acidity"] ).describe()

## 4. Compare groups without ignoring their sizes

Predict which columns remain after grouping. Inspect counts alongside quality summaries, and avoid treating an average based on a few observations as equally stable as one based on many.

A group summary answers a different kind of question from a row filter. Instead of asking which individual wines pass a condition, we ask how a category behaves overall. For example: how many red and white rows are there, and what is each type's average quality?

The operation has three steps: group rows by `wine_type`, calculate summaries inside each group, then return one result row per group.

| wine_type | rows | quality values | Mean quality |
| --- | ---: | --- | ---: |
| red | 2 | 5, 7 | `(5 + 7) / 2 = 6` |
| white | 3 | 6, 6, 8 | `(6 + 6 + 8) / 3 = 6.67` |

This small example shows what aggregation means; the following code calculates the same kind of summaries from every row in the full wine dataset. `pl.len()` counts rows, while `.mean()` computes an average for the selected column.

In [ ]:
type_summary = (
    wine.group_by("wine_type")
    .agg(
        pl.len().alias("rows"),
        pl.col("quality").mean().round(2).alias("mean_quality"),
        pl.col("alcohol").mean().round(2).alias("mean_alcohol"),
        pl.col("volatile acidity").mean().round(3).alias("mean_volatile_acidity"),
    )
    .sort("mean_quality", descending=True)
)
type_summary

In [ ]:
quality_summary = (
    wine.group_by(["wine_type", "quality"])
    .agg(
        pl.len().alias("rows"),
        pl.col("alcohol").mean().round(2).alias("mean_alcohol"),
        pl.col("density").mean().round(4).alias("mean_density"),
    )
    .sort(["wine_type", "quality"])
)
quality_summary

## 5. Inspect the work before collecting a lazy result

Identify which rows and columns are needed before calling collect. Read the plan and compare its filters with your intention. A lazy expression describes a computation; it is not itself evidence that the full result has already been loaded.

A normal `DataFrame` is eager: each operation runs and returns a result right away. A `LazyFrame` records the requested operations first. Polars can inspect that plan and optimize it before `collect()` executes it.

| Mode | What happens after an operation | Useful when |
| --- | --- | --- |
| Eager `DataFrame` | The operation runs immediately | Exploring small or already-loaded data |
| Lazy `LazyFrame` | The operation is added to a plan | Building a multi-step query or scanning larger files |
| `.collect()` | Executes a lazy plan and returns a `DataFrame` | We need the final result in memory |
| `.explain()` | Displays the planned operations | We want to inspect what Polars intends to do |

<div style="background-color:#eaf4fb; border-left:5px solid #2878a5; padding:10px 14px; margin:12px 0;">
<strong>Lazy does not mean automatically faster in every situation.</strong><br>
The optimizer can avoid unnecessary work, especially with `scan_csv` or `scan_parquet` and larger sources. Here the wine data is already in memory, so the example teaches planning and execution rather than claiming a performance gain.
</div>

`explain()` lets us inspect the plan before execution. The next cell filters for high-quality rows, groups them, and then collects only the summary result.

In [ ]:
high_quality_query = (
    wine.lazy()
    .filter(pl.col("quality") >= 7)
    .group_by("wine_type")
    .agg(
        pl.len().alias("high_quality_rows"),
        pl.col("alcohol").mean().round(2).alias("mean_alcohol"),
    )
    .sort("high_quality_rows", descending=True)
)

print(high_quality_query.explain())
high_quality_summary = high_quality_query.collect()
high_quality_summary

## 6. Join summaries back to rows

A join combines tables by matching key values. Here we first calculate one average quality per wine type, then attach that average to every wine row of the same type.

| Row in `wine` | Matching key | Row in `type_baseline` | Value attached |
| --- | --- | --- | --- |
| A red wine with quality 5 | `wine_type = red` | Red average quality | Red average appears beside this wine |
| Another red wine with quality 7 | `wine_type = red` | Same red summary row | The same red average is attached |
| A white wine with quality 6 | `wine_type = white` | White average quality | White average appears beside this wine |

The `how="left"` choice keeps every row from the left table (`wine`) and adds a match from the right table (`type_baseline`). The new difference column is `quality - type_mean_quality`; a negative value means that row's score is below its type's average.

<div style="background-color:#fff5df; border-left:5px solid #d28a16; padding:10px 14px; margin:12px 0;">
<strong>Check key uniqueness before joining.</strong><br>
The summary table has one row per wine type, so each wine row finds one match. If the right table had several rows per key, a join could duplicate left-side rows. That behavior can be correct, but it should be expected.
</div>

In [ ]:
type_baseline = wine.group_by("wine_type").agg(
    pl.col("quality").mean().alias("type_mean_quality")
)

wine_with_baseline = wine.join(type_baseline, on="wine_type", how="left")
wine_with_baseline = wine_with_baseline.with_columns(
    (pl.col("quality") - pl.col("type_mean_quality")).round(2).alias("quality_vs_type_mean")
)
wine_with_baseline.select(["wine_type", "quality", "type_mean_quality", "quality_vs_type_mean"]).head()

## 7. Plot a result

Polars prepares the grouped data; Matplotlib draws the chart. The plot compares the distribution of quality scores by wine type. It helps us see whether the averages hide different shapes or rare scores.

In [ ]:
plot_data = (
    wine.group_by(["wine_type", "quality"])
    .agg(pl.len().alias("rows"))
    .sort(["wine_type", "quality"])
)

plt.figure(figsize=(10, 5))
for wine_type in ["red", "white"]:
    values = plot_data.filter(pl.col("wine_type") == wine_type)
    plt.plot(values["quality"].to_list(), values["rows"].to_list(), marker="o", label=wine_type)
plt.xlabel("Quality score")
plt.ylabel("Number of rows")
plt.title("Wine-quality score distribution")
plt.legend(title="Wine type")
plt.tight_layout()
plt.show()

## 8. Intermediate: inspect types and handle missing values

Before transforming data, inspect the schema. Polars keeps column types explicit, which makes many operations fast and predictable. Missing values are different from zero or an empty string, so choose a policy deliberately: keep them, fill them, or remove the row.

In [ ]:
print(wine.schema)

small_with_missing = pl.DataFrame({"value": [1.0, None, 3.0]})
missing_handled = small_with_missing.with_columns(
    pl.col("value").fill_null(pl.col("value").mean()).alias("value_filled")
)
missing_handled

## 9. Intermediate: build reusable expressions

The main Polars skill is thinking in expressions. `filter` selects rows, `select` chooses columns, `with_columns` creates columns, and `when/then/otherwise` expresses conditional logic. Keep calculations inside the Polars query so the engine can optimize them.

In [ ]:
high_alcohol = (
    wine.filter(pl.col("alcohol") > 12)
    .select(
        pl.col("wine_type"),
        pl.col("quality"),
        pl.col("alcohol").round(1).alias("alcohol_rounded"),
        pl.col("pH").round(2),
    )
    .sort("quality", descending=True)
)
high_alcohol.head()

## 10. Intermediate: window functions

A group aggregation reduces each group to one row. A window expression calculates a group-level value but keeps every original row. This difference matters when we want to compare each observation with its group rather than replace the observations with a summary.

| Method | Rows returned | Example question |
| --- | --- | --- |
| `group_by(...).agg(...)` | One row per group | What is the mean quality for each wine type? |
| `.mean().over("wine_type")` | One value attached to every original row | How does each wine compare with its type's mean? |
| `.rank().over("wine_type")` | One rank per original row within its group | Which quality scores are highest within each type? |

For a tiny example, if red-wine scores are 5 and 7, their group mean is 6. A window calculation keeps both rows and attaches 6 to each; their differences from the mean are -1 and +1. The following code does the same with all the wine rows and also ranks each score within its type.

<div style="background-color:#eaf4fb; border-left:5px solid #2878a5; padding:10px 14px; margin:12px 0;">
<strong>Use aggregation to shrink; use a window to annotate.</strong><br>
This is the fastest way to choose between the two patterns: should the output be a compact summary, or should it preserve every record with extra group context?
</div>

In [ ]:
wine_with_window = wine.with_columns(
    pl.col("quality").mean().over("wine_type").alias("type_mean_quality"),
    pl.col("quality").rank("dense", descending=True).over("wine_type").alias("type_quality_rank"),
)
wine_with_window.select(
    ["wine_type", "quality", "type_mean_quality", "type_quality_rank"]
).sort(["wine_type", "type_quality_rank"]).head(10)

## 11. Intermediate: reshape tables

Long data stores one observation per row and is usually best for filtering and grouping. Wide data spreads categories across columns and can be convenient for reports. `pivot` moves long data to wide form; `unpivot` moves selected columns back to long form.

| Long form: one category per row | Wide form: categories become columns |
| --- | --- |
| quality 5, red: 12 rows | quality 5, red: 12, white: 18 |
| quality 5, white: 18 rows | quality 6, red: 24, white: 30 |
| quality 6, red: 24 rows | |
| quality 6, white: 30 rows | |

The numbers here are a small illustration, not a claim about the downloaded dataset. The key is the layout: long form stores the category in a column; wide form uses a separate column for each category.

<div style="background-color:#eaf4fb; border-left:5px solid #2878a5; padding:10px 14px; margin:12px 0;">
<strong>Keep data long while transforming; make it wide when the question calls for a report.</strong><br>
Long tables work naturally with filters, groups, and new categories. Wide tables can make side-by-side comparisons easier to read, but the category set becomes part of the column structure.
</div>

In [ ]:
quality_by_type = (
    wine.group_by(["quality", "wine_type"])
    .agg(pl.len().alias("rows"))
    .sort(["quality", "wine_type"])
)
quality_wide = quality_by_type.pivot(
    on="wine_type",
    index="quality",
    values="rows",
).fill_null(0)

print("Wide report:")
print(quality_wide)
print("Long report again:")
quality_wide.unpivot(index="quality", variable_name="wine_type", value_name="rows").head()

## 12. Intermediate: work with dates and times

Polars has a temporal type system and `.dt` expressions for date parts and durations. Parse dates early, keep the column as a date or datetime, and avoid doing calendar logic on raw strings.

In [ ]:
dated_rows = pl.DataFrame(
    {
        "measured_at": ["2024-01-15", "2024-02-15", "2025-01-15"],
        "value": [10.0, 12.0, 15.0],
    }
).with_columns(
    pl.col("measured_at").str.to_date(),
)

dated_rows.with_columns(
    pl.col("measured_at").dt.year().alias("year"),
    pl.col("measured_at").dt.month().alias("month"),
)

## 13. Advanced: file formats and lazy query planning

CSV is convenient for interchange, but Parquet preserves types and is usually better for repeated analytical work. `scan_csv` and `scan_parquet` build lazy plans without loading the whole source immediately. Projection and predicate pushdown let Polars read or process only what the query needs.

In [ ]:
parquet_buffer = io.BytesIO()
wine.select(["wine_type", "quality", "alcohol"]).head(100).write_parquet(parquet_buffer)
parquet_buffer.seek(0)
round_trip = pl.read_parquet(parquet_buffer)
print(f"Parquet round-trip rows: {round_trip.height}")

optimized_query = (
    wine.lazy()
    .select(["wine_type", "quality", "alcohol"])
    .filter(pl.col("quality") >= 7)
    .group_by("wine_type")
    .agg(pl.col("alcohol").mean().alias("mean_alcohol"))
)
print(optimized_query.explain())

## 14. Advanced: SQL and interoperability

Polars expressions are the main API, but `SQLContext` can help teams who already think in SQL. Keep the query language choice consistent inside a pipeline, and remember that SQL does not remove the need to understand types, nulls, joins, or grouping semantics.

In [ ]:
sql_context = pl.SQLContext(wine=wine)
sql_summary = sql_context.execute(
    """
    SELECT wine_type, AVG(quality) AS mean_quality, COUNT(*) AS rows
    FROM wine
    GROUP BY wine_type
    ORDER BY mean_quality DESC
    """,
    eager=True,
)
sql_summary

## 15. Check the transformation against a known small result

Before reading the assertion, calculate what the small example should produce. If a test fails, distinguish changed business meaning from changed implementation. Preserve checks on schema, nulls and row counts when replacing input files.

Data pipelines need tests, not only attractive output. `polars.testing.assert_frame_equal` checks that a transformation returns the expected rows, columns, and values. Small assertions catch accidental changes in joins, filters, and column types.

In [ ]:
from polars.testing import assert_frame_equal

expected = pl.DataFrame({"quality": [5, 5], "wine_type": ["red", "red"]})
actual = wine.filter(pl.col("wine_type") == "red").select(["quality", "wine_type"]).head(2)
assert_frame_equal(actual, expected)
print("Transformation test passed")

## 16. Advanced: performance and Python functions

Prefer native Polars expressions because they can be optimized and run in Polars' engine. `map_elements` runs a Python function element by element and can be much slower; use it only when the operation cannot be expressed with Polars expressions, and specify the return type when needed.

In [ ]:
native_result = wine.select(
    (pl.col("alcohol") * 2).alias("alcohol_doubled")
).head()
native_result

## Explain what the evidence supports

Compare a wine-quality group using its count and summary statistics. Explain one possible confounder and one query-plan change that could reduce work without changing the answer.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: audit a messy export before collecting an aggregate

A payment export contains retries, invalid amounts, malformed dates and unknown customer keys. Refunds are signed negative amounts, so deleting every negative amount would inflate net receipts. Reconcile 6,060 raw rows into 60 retries, 50 rejected events and 5,950 usable events. These are generated operational-style records, not actual customer payments. The generator's clean source is used only as a verification oracle, never to fill invalid source fields. Work within a small laptop-sized budget, then identify which intermediate tables or buffers would grow at scale.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through strict versus tolerant casting

Casting `['120', 'bad', '-20']` directly to integers should fail. A tolerant cast retains the two usable values and marks the malformed field null. The signed total is 100, not 120: a refund is valid data. Count nulls before dropping rows; otherwise a successful query hides the ingestion failure.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

p2_rng=np.random.default_rng(91)
p2_n=6000
p2_source=pd.DataFrame({'event_id':np.arange(p2_n),'customer_id':p2_rng.integers(1,121,p2_n),
    'day':pd.Timestamp('2025-01-01')+pd.to_timedelta(p2_rng.integers(0,90,p2_n),unit='D'),
    'amount_cents':p2_rng.integers(100,100000,p2_n)})
p2_source.loc[p2_source.event_id%17==0,'amount_cents'] *= -1 # Signed refunds are valid.
p2_raw=p2_source.astype(str)
p2_raw.loc[0:19,'amount_cents']='bad'
p2_raw.loc[20:29,'amount_cents']=''
p2_raw.loc[30:37,'day']='not-a-date'
p2_raw.loc[38:49,'customer_id']='9999'
p2_raw=pd.concat([p2_raw,p2_raw.iloc[-60:]],ignore_index=True)
p2_customers=pd.DataFrame({'customer_id':np.arange(1,121),'region':np.tile(['East','West','North','South'],30)})
p2_raw.to_csv(p2_out/'ledger_raw.csv',index=False)
# Independent generator-based expected population; never repair a bad value from this oracle.
p2_expected=p2_source.iloc[50:]
p2_expected_total=int(p2_expected.amount_cents.sum())
assert len(p2_raw)==6060 and len(p2_expected)==5950

import polars as pl
p2_demo=pl.DataFrame({'amount':['120','bad','-20']}).with_columns(pl.col('amount').cast(pl.Int64,strict=False).alias('cents'))
assert p2_demo['cents'].null_count()==1 and p2_demo['cents'].sum()==100
display(p2_demo)
p2_lf=pl.scan_csv(p2_out/'ledger_raw.csv',infer_schema=False).unique()
p2_lf=p2_lf.with_columns(pl.col('amount_cents').cast(pl.Int64,strict=False).alias('cents'),
    pl.col('customer_id').cast(pl.Int64,strict=False),pl.col('day').str.to_date(strict=False).alias('parsed_day'))
p2_valid_expr=pl.col('cents').is_not_null()&pl.col('parsed_day').is_not_null()&pl.col('customer_id').is_between(1,120)
p2_clean_lf=p2_lf.filter(p2_valid_expr)
p2_customers_pl=pl.from_pandas(p2_customers)
p2_report_plan=(p2_clean_lf.join(p2_customers_pl.lazy(),on='customer_id',validate='m:1')
 .group_by('region').agg(pl.len().alias('events'),pl.col('cents').sum().alias('net_cents')).sort('region'))
print(p2_report_plan.explain())

### Work through eager versus lazy execution

Collecting raw rows first and then aggregating is easy to inspect. Pushing selection and aggregation into a lazy plan can avoid materializing unnecessary output columns. Exact duplicate removal still needs global state; streaming is not a promise that every operation has constant memory. Compare the final sorted tables, then inspect the plan and collected result sizes.

In [ ]:
p2_t=time.perf_counter();p2_lazy_report=p2_report_plan.collect();p2_lazy_time=time.perf_counter()-p2_t
p2_t=time.perf_counter()
p2_eager=p2_clean_lf.collect().join(p2_customers_pl,on='customer_id',validate='m:1')
p2_eager_report=p2_eager.group_by('region').agg(pl.len().alias('events'),pl.col('cents').sum().alias('net_cents')).sort('region')
p2_eager_time=time.perf_counter()-p2_t
from polars.testing import assert_frame_equal
assert_frame_equal(p2_lazy_report,p2_eager_report)
assert p2_lazy_report['events'].sum()==5950 and p2_lazy_report['net_cents'].sum()==p2_expected_total
assert p2_lf.filter(~p2_valid_expr).select(pl.len()).collect().item()==50
p2_lazy_report.write_csv(p2_out/'polars_payment_report.csv')
display(p2_lazy_report)
print({'lazy_seconds':p2_lazy_time,'eager_seconds':p2_eager_time,'collected_detail_bytes':p2_eager.estimated_size(),'report_bytes':p2_lazy_report.estimated_size()})

### Explain the resource claim precisely

The result has four regional rows, while the eager detail has 5,950. That difference proves less output was materialized, not that peak process memory fell by exactly that ratio. The timing includes cache effects and is not a rigorous engine benchmark.

**Try:** select only January before aggregating. **Worked reasoning:** place the date predicate in the lazy pipeline, then read the optimized plan. Keep the same date predicate in the eager comparison. Comparing different populations is not evidence of an optimization. If the customer table develops duplicate keys, keep join validation and investigate the data rather than removing the check.